# COMP560 ReID — Kaggle Runner

Kaggle has no Google Drive. We pull datasets via `gdown` (share-link file IDs), train, then upload checkpoints back to a Drive folder via a short-lived service-account (or just download the notebook output).

Fill in the **Config** cell, then Run All. Turn on GPU (T4 / P100).

In [ ]:
# ====== Config ======
# Per-run, change CONFIG_NAME (and set TEACHER_CKPT_ID for KD runs).
#
# Teacher runs (no KD):   CONFIG_NAME = 'exp01_teacher_dinov2_large'  (or exp02)
#                         TEACHER_CKPT_ID = ''
# KD runs (student):      CONFIG_NAME = 'exp05_student_resnet50_kd'   (or exp06)
#                         TEACHER_CKPT_ID = '1KhOhhyFaPLSF4snWLwAcp1daHsqGQa4I'
#                                           ^ exp01 teacher (DinoV2-L) slim checkpoint
CONFIG_NAME   = 'exp01_teacher_dinov2_large'
REPO_URL      = 'https://github.com/AlexTourneux/ReID_final.git'
BRANCH        = 'main'

# Google Drive file IDs (get from right-click > Get link on each file)
DATASET_A_TAR_ID  = '1QedUDFWFzUAKmGd23EbL3ugYud0MKorp'
MARKET1501_ZIP_ID = '12llGoZDgbBq6BRnyu27VuDj78bksBDDl'

# For KD runs — Drive file id of the teacher's best.pth (slim is fine), or leave blank.
# Known teacher IDs:
#   exp01 DinoV2-Large:  1KhOhhyFaPLSF4snWLwAcp1daHsqGQa4I
TEACHER_CKPT_ID   = ''

WORKDIR = '/kaggle/working/object-reid'
print('CONFIG_NAME:', CONFIG_NAME)
print('KD run:     ', bool(TEACHER_CKPT_ID))


In [ ]:
# ====== Clone repo ======
import subprocess, os, shutil
if os.path.isdir(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, WORKDIR])
os.chdir(WORKDIR)
!git log -1 --oneline

In [ ]:
# ====== Install deps ======
!pip install -q -r requirements.txt

In [ ]:
# ====== Download image archives via gdown ======
# Parquets are already in the cloned repo. Only the image archives are on Drive.
import os, gdown
os.makedirs('datasets/dataset_a', exist_ok=True)
os.makedirs('datasets/market1501', exist_ok=True)

def gdown_file(fid, out):
    if fid and not os.path.exists(out):
        gdown.download(f'https://drive.google.com/uc?id={fid}', out, quiet=False)

gdown_file(DATASET_A_TAR_ID,  'datasets/dataset_a/images.tar.gz')
gdown_file(MARKET1501_ZIP_ID, 'datasets/market1501/market1501.zip')

if not os.path.isdir('datasets/dataset_a/images'):
    !tar -xzf datasets/dataset_a/images.tar.gz -C datasets/dataset_a
if not os.path.isdir('datasets/market1501/bounding_box_train'):
    !unzip -q datasets/market1501/market1501.zip -d datasets/market1501


In [ ]:
# ====== (Optional) pull teacher checkpoint for KD runs ======
import os, gdown
if TEACHER_CKPT_ID:
    teacher_dst = 'checkpoints/exp01_teacher_dinov2_large/best.pth'
    os.makedirs(os.path.dirname(teacher_dst), exist_ok=True)
    gdown.download(f'https://drive.google.com/uc?id={TEACHER_CKPT_ID}', teacher_dst, quiet=False)
    print('Teacher loaded:', teacher_dst)

In [ ]:
# ====== Train ======
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python -u train.py --config configs/experiments/{CONFIG_NAME}.yaml

In [ ]:
# ====== Save outputs to /kaggle/working ======
# Kaggle sessions persist output under /kaggle/working. Download it after
# the run and upload to Drive manually, OR use the 'Save Version' output.
import os, shutil
out_dir = f'/kaggle/working/ckpt_out/{CONFIG_NAME}'
os.makedirs(os.path.dirname(out_dir), exist_ok=True)
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir)
shutil.copytree(f'checkpoints/{CONFIG_NAME}', out_dir)
!ls -lh "$out_dir"